# This notebook generate raw plot used in Figure 2a

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import numpy as np

plt.rcParams["font.family"] = "Arial"
sns.set_theme(style="darkgrid")

Load MaskRCNN Learning curves

In [ ]:
mask_path = '../../../datastore/results/MaskRCNNFinal.pkl'

with open(mask_path, 'rb') as f:
    data = pickle.load(f)

data = data['MaskRCNN']
f1s = data['f1s']
precisions = data['precisions']
recalls = data['recalls']
n_epochs = len(f1s)
epochs = np.arange(1, 51, 1)

idx = np.argmax(f1s)
print(f'Best epoch ({idx}) f1 = {f1s[idx]} \n best epoch prec = {precisions[idx]} \n best epoch recalls = {recalls[idx]}')

Load veto classifier learning curves

In [ ]:
# these were initially parsed by ChatGPT from cell outputs because my runtime disconnected before i could save the results properly
veto_path = '../../../datastore/results/veto_learning.pkl' 

with open(veto_path, 'rb') as f:
    veto_data = pickle.load(f)

veto_f1s = veto_data['F1 Score']
veto_precisions = veto_data['Precision']
veto_recalls = veto_data['Recall']

Smooth with exponentially weighted moving averages

In [ ]:
smoothed_f1 = pd.Series(f1s).ewm(span=3, adjust=False).mean()
smoothed_p = pd.Series(precisions).ewm(span=3, adjust=False).mean()
smoothed_r = pd.Series(recalls).ewm(span=3, adjust=False).mean()

smoothed_vf1 = pd.Series(veto_f1s).ewm(span=3, adjust=False).mean()
smoothed_vp = pd.Series(veto_precisions).ewm(span=3, adjust=False).mean()
smoothed_vr = pd.Series(veto_recalls).ewm(span=3, adjust=False).mean()

In [ ]:
palette = {
    "f1": {
        "color": "#2166ac",
        "linestyle": "-",
        "linewidth": 1.8,
        "alpha": 1.0
    },
    "precision": {
        "color": "#67a9cf",
        "linestyle": "--",
        "linewidth": 1.2,
        "alpha": 0.8
    },
    "recall": {
        "color": "#d1e5f0",
        "linestyle": "-.",
        "linewidth": 1.2,
        "alpha": 0.8
    },

    "vf1": {
        "color": "#b2182b",
        "linestyle": "-",
        "linewidth": 1.8,
        "alpha": 1.0
    },
    "vprecision": {
        "color": "#ef8a62",
        "linestyle": "--",
        "linewidth": 1.2,
        "alpha": 0.8
    },
    "vrecall": {
        "color": "#fddbc7",
        "linestyle": "-.",
        "linewidth": 1.2,
        "alpha": 0.8
    },
}

TICK_FS = 11
# maskRCNN
fig, ax = plt.subplots(figsize=(3.7, 4.73), dpi=150)

ax.plot(epochs, smoothed_p, label="Precision", **palette["precision"])

ax.plot(epochs, smoothed_r, label="Recall",**palette["recall"])

ax.plot(epochs, smoothed_f1, label="F1 Score", **palette["f1"])

ax.set(xlim=(1, 50), ylim=(0, 1))

ax.set_xticks(np.arange(0, 51, 10))
ax.set_yticks(np.arange(0, 1.01, 0.2))


# VetoClassifier
ax.plot(epochs, smoothed_vp, label="Precision", **palette["vprecision"])
ax.plot(epochs, smoothed_vr, label="Recall",**palette["vrecall"])
ax.plot(epochs, smoothed_vf1, label="F1 Score", **palette["vf1"])
ax.set( xlabel="Epoch", ylabel="Metric value", xlim=(1, 50), ylim=(0, 1))
ax.set_xticks(np.arange(0, 51, 10))
ax.set_yticks(np.arange(0, 1.01, 0.2))




ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.45)
ax.grid(axis="x", visible=False)

ax.legend(
    frameon=False,
    loc="lower center",
    ncol=1,
    fontsize=TICK_FS,
)

ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    bottom=True,
    left=True,
    top=False,
    right=False,
    length=4,
    width=0.8,
    color="black",
    labelcolor="black",
    labelsize=TICK_FS
)

ax.spines["bottom"].set_visible(True)
ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_color("black")
ax.spines["left"].set_color("black")
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
ax.spines["top"].set_color("black")
ax.spines["right"].set_color("black")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Metric value", fontsize=12)

fig.tight_layout()

plt.savefig('out/Fig2a.svg', dpi=400)